# Lab | Convolutional Neural Networks

In this lab we are going to explore how Convolutional Neural Networks (CNNs) work, starting from understanding what a convolution actually does at the pixel level, all the way to training a real classifier on the CIFAR-10 dataset and experimenting with data augmentation.

I'm honestly pretty excited about this one because we finally get to work with real images instead of tabular data!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print(f"Using device: {device}")

---
## Task 1 — Convolution Mechanics: Filters and Shapes

Before jumping into training, I want to understand what a convolution is actually doing to the image. The idea is that a filter (a small matrix of weights) slides across the image and produces a new feature map. Depending on what values are in the filter, it can detect edges, blur the image, sharpen it, etc.

### Part A — Handcrafted Filters

First I'll load a single CIFAR-10 image and apply three manually defined filters to it:
- A **vertical edge detector** — highlights vertical lines/edges
- A **horizontal edge detector** — highlights horizontal lines/edges
- A **blur filter** — smooths out the image by averaging neighbouring pixels

In [ ]:
# Load CIFAR-10 and grab a single image
raw_dataset = datasets.CIFAR10(root='./data', train=True, download=True,
                                transform=transforms.ToTensor())

image, label = raw_dataset[0]
class_names = ['airplane','automobile','bird','cat','deer',
                'dog','frog','horse','ship','truck']

print(f"Image shape: {image.shape}")
print(f"Label: {class_names[label]}")

# Add batch dimension -> (1, 3, 32, 32)
img_input = image.unsqueeze(0)

In [ ]:
def apply_handcrafted_filter(img_tensor, kernel_2d):
    """
    Applies a single 2D kernel to all 3 input channels and sums the result.
    Returns the output as a 2D numpy array for plotting.
    """
    conv = nn.Conv2d(3, 1, kernel_size=3, padding=1, bias=False)
    # Stack the same kernel for each of the 3 input channels
    kernel_tensor = torch.tensor(kernel_2d, dtype=torch.float32)
    kernel_tensor = kernel_tensor.unsqueeze(0).unsqueeze(0)  # (1,1,3,3)
    kernel_tensor = kernel_tensor.repeat(1, 3, 1, 1)         # (1,3,3,3)
    conv.weight = nn.Parameter(kernel_tensor)
    with torch.no_grad():
        out = conv(img_tensor)  # (1,1,32,32)
    return out.squeeze().numpy()


# Define the three kernels
vertical_kernel   = [[-1, 0, 1],
                     [-1, 0, 1],
                     [-1, 0, 1]]

horizontal_kernel = [[-1,-1,-1],
                     [ 0, 0, 0],
                     [ 1, 1, 1]]

blur_kernel       = (np.ones((3, 3)) / 9).tolist()

# Apply each filter
out_vertical   = apply_handcrafted_filter(img_input, vertical_kernel)
out_horizontal = apply_handcrafted_filter(img_input, horizontal_kernel)
out_blur       = apply_handcrafted_filter(img_input, blur_kernel)

# Plot: original + 3 filtered versions
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(image.permute(1, 2, 0).numpy())
axes[0].set_title(f"Original\n({class_names[label]})")
axes[0].axis('off')

axes[1].imshow(out_vertical, cmap='gray')
axes[1].set_title("Vertical Edge\nDetector")
axes[1].axis('off')

axes[2].imshow(out_horizontal, cmap='gray')
axes[2].set_title("Horizontal Edge\nDetector")
axes[2].axis('off')

axes[3].imshow(out_blur, cmap='gray')
axes[3].set_title("Blur Filter")
axes[3].axis('off')

plt.suptitle("Handcrafted Convolution Filters on a CIFAR-10 Image", fontsize=13)
plt.tight_layout()
plt.show()

#### What each filter highlights

- **Vertical edge detector** `[[-1,0,1],[-1,0,1],[-1,0,1]]`: This filter responds strongly wherever there's a sharp change in pixel intensity going *left to right*. Looking at the output, I can clearly see the vertical outlines/contours of the object. Flat regions with uniform colour appear dark because there's no change to detect.

- **Horizontal edge detector** (transpose of the above): Same idea but rotated 90°. It picks up changes going *top to bottom*, so horizontal boundaries and edges light up. The roof line, ground line, or any horizontal structure shows up clearly.

- **Blur filter** `(1/9) * ones(3,3)`: This is just a simple averaging filter — each output pixel becomes the average of its 3×3 neighbourhood. The result is a softer, smoother version of the original. Fine detail and noise are reduced. This is useful as a preprocessing step to reduce noise before edge detection.

**Key takeaway:** Even without any learning, carefully chosen filters can extract meaningful features. CNNs essentially *learn* which filters are most useful for the task at hand — which is really elegant.

### Part B — Shape Tracking

Now I want to understand how the spatial dimensions change as data flows through a CNN. I'll build a `TinyCNN` block and track the shape after each operation.

The formula for the output size of a Conv2d layer is:
$$\text{output\_size} = \left\lfloor \frac{\text{input\_size} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1$$

MaxPool2d(2) simply halves the spatial dimensions.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)

# Dummy batch: 8 images, 3 channels, 32x32
x = torch.randn(8, 3, 32, 32)

model_tiny = TinyCNN()

print(f"Input:          {x.shape}")
x = model_tiny.conv1(x);  print(f"After conv1:    {x.shape}")
x = model_tiny.pool1(x);  print(f"After pool1:    {x.shape}")
x = model_tiny.conv2(x);  print(f"After conv2:    {x.shape}")
x = model_tiny.pool2(x);  print(f"After pool2:    {x.shape}")

#### Shape-Tracking Table

| Layer  | Input shape        | Output shape       |
|--------|--------------------|--------------------||
| conv1  | (8, 3, 32, 32)     | (8, 16, 32, 32)    |
| pool1  | (8, 16, 32, 32)    | (8, 16, 16, 16)    |
| conv2  | (8, 16, 16, 16)    | (8, 32, 16, 16)    |
| pool2  | (8, 32, 16, 16)    | (8, 32, 8, 8)      |

A few things I notice:
- `Conv2d` with `padding=1` and `kernel_size=3` **preserves** the spatial dimensions (32→32, 16→16). The padding compensates for the kernel overlap at the borders.
- `MaxPool2d(2)` **halves** the spatial dimensions each time (32→16→8).
- The number of **channels** (depth) increases: 3 → 16 → 32. This is how CNNs trade spatial resolution for richer feature representations.
- After two pooling layers we end up with 32 feature maps of size 8×8, giving us 32×8×8 = 2048 values to feed into a classifier.

---
## Task 2 — Train a Small CNN on CIFAR-10

Now for the real thing. I'll build a more complete CNN with two convolutional blocks (each with BatchNorm for stable training) and a small fully-connected classifier head. Then I'll train it on CIFAR-10 for 15 epochs and plot the learning curves.

BatchNorm is really useful here because it normalises the activations after each convolution, which helps with gradient flow and lets us use a higher learning rate without things blowing up.

In [ ]:
# ── Data loaders (no augmentation baseline) ──────────────────────────────────
basic_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

train_set_basic = datasets.CIFAR10('./data', train=True,  download=True,  transform=basic_tf)
val_set         = datasets.CIFAR10('./data', train=False, download=True,  transform=basic_tf)

train_loader_basic = DataLoader(train_set_basic, batch_size=128, shuffle=True,  num_workers=2)
val_loader         = DataLoader(val_set,         batch_size=128, shuffle=False, num_workers=2)

print(f"Training samples:   {len(train_set_basic):,}")
print(f"Validation samples: {len(val_set):,}")

In [ ]:
# ── Model definition ─────────────────────────────────────────────────────────
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 32x32 -> 16x16
        )

        # Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 16x16 -> 8x8
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x


# Count parameters
model = CIFAR10CNN().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
print(model)

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [],
                'train_acc':  [], 'val_acc':  []}

    for epoch in range(1, epochs + 1):
        # ── Training phase
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += images.size(0)

        train_loss = running_loss / total
        train_acc  = 100.0 * correct / total

        # ── Validation phase
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss_sum += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total   += images.size(0)

        val_loss = val_loss_sum / val_total
        val_acc  = 100.0 * val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

    return history


print("Training baseline model (no augmentation)...")
model_baseline = CIFAR10CNN().to(device)
history_baseline = train_model(model_baseline, train_loader_basic, val_loader, epochs=15)

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
def plot_history(history, title="Training Curves"):
    epochs = range(1, len(history['train_loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.plot(epochs, history['train_loss'], label='Train Loss', marker='o', markersize=4)
    ax1.plot(epochs, history['val_loss'],   label='Val Loss',   marker='s', markersize=4)
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Cross-Entropy Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(epochs, history['train_acc'], label='Train Acc', marker='o', markersize=4)
    ax2.plot(epochs, history['val_acc'],   label='Val Acc',   marker='s', markersize=4)
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


plot_history(history_baseline, title="Task 2 — Baseline (No Augmentation)")

best_val_baseline = max(history_baseline['val_acc'])
train_val_gap_baseline = history_baseline['train_acc'][-1] - history_baseline['val_acc'][-1]
print(f"Best val accuracy (baseline): {best_val_baseline:.2f}%")
print(f"Train/val gap at final epoch:  {train_val_gap_baseline:.2f}%")

#### Task 2 Observations

The model trained pretty smoothly. As expected, training accuracy is noticeably higher than validation accuracy — this is a sign of **overfitting**: the model is memorising some training patterns that don't generalise well to unseen images.

The validation accuracy plateaus somewhere around 70–75%, which matches the expected behaviour mentioned in the lab. Not bad for a small network trained on CPU!

The train/val gap is the main thing to address in Task 3.

---
## Task 3 — Data Augmentation

The idea behind data augmentation is simple: instead of showing the model the exact same image every epoch, we randomly transform it — flip it, crop it slightly, adjust the brightness — so the model sees a slightly different version each time. This artificially inflates the effective size of our training set and forces the model to learn more robust, generalised features rather than memorising specific pixel patterns.

Importantly, augmentation is applied **only to the training data**. The validation set stays unchanged so our accuracy measurement stays fair and consistent.

Let's see if this closes the train/val gap from Task 2.

In [ ]:
# ── Augmented training loader ─────────────────────────────────────────────────
train_tf_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

train_set_aug = datasets.CIFAR10('./data', train=True, download=False, transform=train_tf_aug)
train_loader_aug = DataLoader(train_set_aug, batch_size=128, shuffle=True, num_workers=2)

# val_loader remains the same as Task 2 (plain ToTensor + Normalize)
print("Augmented training loader created.")
print(f"Training samples: {len(train_set_aug):,}  (same count, different transforms each epoch)")

In [ ]:
# ── Quick visualisation of augmentation effect ────────────────────────────────
# Show the same image before and after augmentation to see what's happening
raw_img, _ = raw_dataset[0]   # original (ToTensor only, no norm)

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
axes[0].imshow(raw_img.permute(1, 2, 0).numpy())
axes[0].set_title('Original')
axes[0].axis('off')

aug_only = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
])

from PIL import Image as PILImage
raw_pil = PILImage.fromarray((raw_img.permute(1,2,0).numpy() * 255).astype(np.uint8))

for i in range(1, 6):
    aug_img = transforms.ToTensor()(aug_only(raw_pil))
    axes[i].imshow(aug_img.permute(1, 2, 0).numpy())
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')

plt.suptitle("Effect of Data Augmentation (5 random variations)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Train the same architecture with augmentation ─────────────────────────────
print("Training augmented model...")
model_augmented = CIFAR10CNN().to(device)
history_augmented = train_model(model_augmented, train_loader_aug, val_loader, epochs=15)

In [ ]:
# ── Plot augmented model curves ───────────────────────────────────────────────
plot_history(history_augmented, title="Task 3 — With Data Augmentation")

best_val_aug = max(history_augmented['val_acc'])
train_val_gap_aug = history_augmented['train_acc'][-1] - history_augmented['val_acc'][-1]
print(f"Best val accuracy (augmented): {best_val_aug:.2f}%")
print(f"Train/val gap at final epoch:   {train_val_gap_aug:.2f}%")

In [ ]:
# ── Side-by-side comparison plot ──────────────────────────────────────────────
epochs = range(1, 16)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(epochs, history_baseline['val_loss'],   label='No Augmentation', marker='o', markersize=4)
ax1.plot(epochs, history_augmented['val_loss'],  label='With Augmentation', marker='s', markersize=4)
ax1.set_title('Validation Loss Comparison')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, history_baseline['val_acc'],   label='No Augmentation', marker='o', markersize=4)
ax2.plot(epochs, history_augmented['val_acc'],  label='With Augmentation', marker='s', markersize=4)
ax2.set_title('Validation Accuracy Comparison')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle("Task 2 vs Task 3 — Augmentation Impact", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("="*60)
print(f"{'Run':<30} {'Best Val Acc':>14} {'Train/Val Gap':>14}")
print("-"*60)
print(f"{'Task 2 (no augmentation)':<30} {best_val_baseline:>13.2f}% {train_val_gap_baseline:>13.2f}%")
print(f"{'Task 3 (with augmentation)':<30} {best_val_aug:>13.2f}% {train_val_gap_aug:>13.2f}%")
print("="*60)

#### Task 3 Analysis — What Changed With Augmentation?

Looking at the results table and the comparison plots, a few things stand out:

**1. The train/val gap shrank significantly.**
Without augmentation, the model had a noticeably larger gap between training and validation accuracy — classic overfitting. With augmentation, the model can no longer simply memorise the exact training images (since each epoch shows slightly different versions), so it's forced to learn more general features. The gap closes.

**2. Validation accuracy improved.**
The augmented model typically reaches a better peak validation accuracy. Even though training accuracy might be a bit lower (because the augmented images are harder), the model generalises better to the clean validation set.

**3. The training curves are smoother.**
Augmentation acts as a mild regulariser. The loss curves tend to be less noisy and the model doesn't overfit as aggressively in later epochs.

**Why does this work intuitively?**
- `RandomCrop`: forces the model to recognise objects regardless of their exact position in the frame.
- `RandomHorizontalFlip`: makes the model invariant to left-right orientation (a cat facing left is still a cat).
- `ColorJitter`: teaches the model to focus on shape and structure rather than exact colour values.

**Conclusion:** Data augmentation is a free lunch — same dataset, same architecture, same training time, but meaningfully better generalisation. This is why it's basically standard practice in any image classification pipeline.